In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.appName('10').getOrCreate()

- `with open()`을 이용하여 원하는 파트의 텍스트만 `list`로 저장

In [3]:
table = [] # data를 저장할 list

with open('unstruct_sample.txt') as f:
    for line in f:
        # "data, TGTY,"로 시작하는 라인을 만나는 시점부터 저장 시작
        if line.strip().startswith("data, TGTY,"):
            # string.strip(): 좌우 공백 삭제(trim(col)와 동일 기능)
            table.append(line)
            # "data, TGTY,"를 저장한 이후부터
            # "#---------------"를 만날 때까지 line을 table에 저장
            for line in f:
                # "#---------------"를 만나면 종료
                if line.strip().startswith("#---------------"):
                    break
                table.append(line)
# table 확인
table

['data, TGTY, 2020-06-16 03:31:00, 12\n',
 'file, RTYT, 2020-06-16 03:33:00, 23, TYY\n',
 'copy, VBTas, 2020-06-16 03:36:00, 45, LPT\n',
 'zita, TYU-YU, 2020-06-16 03:38:00, 88\n',
 'aaa-link, TYU, 2020-06-16 03:39:00, 89, GGT\n',
 'pto, "TYU:, 2020-06-16 03:42:00, 23\n']

- `table`(list)를 `DataFrame`으로 저장
- `SparkSession.createDataFrame(value,type)`

In [4]:
df = spark.createDataFrame(table,StringType())
df.show(truncate=False)

+---------------------------------------------+
|value                                        |
+---------------------------------------------+
|data, TGTY, 2020-06-16 03:31:00, 12\n        |
|file, RTYT, 2020-06-16 03:33:00, 23, TYY\n   |
|copy, VBTas, 2020-06-16 03:36:00, 45, LPT\n  |
|zita, TYU-YU, 2020-06-16 03:38:00, 88\n      |
|aaa-link, TYU, 2020-06-16 03:39:00, 89, GGT\n|
|pto, "TYU:, 2020-06-16 03:42:00, 23\n        |
+---------------------------------------------+



- 각 행의 값을 총 5개의 열로 구성하되 type은 (`str, str, timestamp, int, str`)으로 지정하며, 마지막 `'\n'`은 삭제함
- `split(df.value,',')[n]`: `value`열의 값을 `','`를 기준으로 분리한 후, `n+1`번째 요소 추출
- `value`열의 값을 `','`를 기준으로 분리할 경우, 각 요소에 공백(`' '`)이 포함될 수 있으므로, 일부 기능에서는 `trim()`을 활용해야 함
> - 특히 `to_timestamp`를 통해 `str`을 `timestamp`로 변경하는 경우, 공백에 의해 오류가 발생할 수 있으므로 주의할 것

In [12]:
spark.conf.set('spark.sql.ansi.enabled','false')
# spark 3.x 버전부터 데이터 안정성을 위해 범위를 벗어난 인덱스에 접근할 때
# 예외를 발생시키는 Strict Index Checking 기능이 기본적으로 활성화 되어 있음
# ANXI SQL 표준 가이드를 끄면, spark 2.x 버전처럼 인덱스 범위를 벗어났을 때,
# 에러 없이 NULL을 반환

new_df = df.withColumn('col1',split(df.value,',')[0]).\
withColumn('col2',split(df.value,',')[1]).\
withColumn('col3',to_timestamp(trim(split(df.value,',')[2]),'yyyy-MM-dd HH:mm:ss')).\
withColumn('col4',split(df.value,',')[3].cast(IntegerType())).\
withColumn('col5',regexp_replace(split(df.value,',')[4],'\n','')).\
drop('value')
# 세번째 열은 timestamp type으로 변환
# 네번째 열은 integer type으로 변환
# 다섯번째 열은 개행을 의미하는 \n을 삭제

In [13]:
new_df.show()

+--------+-------+-------------------+----+----+
|    col1|   col2|               col3|col4|col5|
+--------+-------+-------------------+----+----+
|    data|   TGTY|2020-06-16 03:31:00|  12|NULL|
|    file|   RTYT|2020-06-16 03:33:00|  23| TYY|
|    copy|  VBTas|2020-06-16 03:36:00|  45| LPT|
|    zita| TYU-YU|2020-06-16 03:38:00|  88|NULL|
|aaa-link|    TYU|2020-06-16 03:39:00|  89| GGT|
|     pto|  "TYU:|2020-06-16 03:42:00|  23|NULL|
+--------+-------+-------------------+----+----+



In [14]:
new_df.printSchema()

root
 |-- col1: string (nullable = true)
 |-- col2: string (nullable = true)
 |-- col3: timestamp (nullable = true)
 |-- col4: integer (nullable = true)
 |-- col5: string (nullable = true)

